# Raman（G / 2D 峰、背景）

和軟體用的是**同一份分析程式**（`magic/`），結果完全一樣；這裡適合自由探索、試新的分析。

- 第一格把 `DATA` 改成你的資料夾（沒填就用合成的範例資料）
- `run(機台, 分析名稱, [檔案], 參數=值…)` 回傳 `(fig, res)`：`res.text` 文字結果、`res.tables` 表格（pandas）
- 參數名稱和軟體參數面板一樣，可以用 `list_analyses("Raman")` 看有哪些分析
- **上傳 GitHub 前會自動清掉輸出**（publish.bat），圖和數值不會被公開


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # 沒有 pip install 也能 import magic（notebooks 的上一層就是工具資料夾）
from magic.runner import run, list_analyses
from magic.readers import load
from magic.core.plotstyle import apply
apply()                                          # 中文字型、配色和軟體一樣

DATA = r""   # ← Raman 資料夾，例如 r"D:\Yeep\01 Magic Tool\Raman\Data"
if not DATA:                                     # 沒填就用範例（合成資料）
    from magic.core.settings import data_dir
    from magic.testdata import write_all
    write_all(os.path.join(str(data_dir()), "demo"))
    DATA = os.path.join(str(data_dir()), "demo", "Raman")
print("資料夾：", DATA)
print("可用的分析：", list_analyses("Raman"))
sorted(os.listdir(DATA))

## 整個資料夾的峰值總表（取代原本的快速版 cell）

In [ ]:
from magic.readers import detect
spectra = [os.path.join(DATA, f) for f in sorted(os.listdir(DATA)) if detect(os.path.join(DATA, f)) == "Raman"]
fig, res = run("Raman", "多檔峰值總表", spectra)
res.tables["peak_summary"]

## 一個檔案的細節：背景、Lorentzian 擬合

有 hBN 的樣品加 `hbn=True`；峰的範圍可以改 `G_win`、`D2_win`、`hBN_win`。

In [ ]:
fig, res = run("Raman", "石墨烯 G / 2D 峰", [spectra[0]], hbn=True, baseline="多項式（只用無峰區）")
print(res.text)
fig

## 背景扣除：比較三種方法，取得扣背景後的光譜

In [ ]:
fig, res = run("Raman", "背景扣除", [spectra[0]], method="ALS（自動）")
corrected = list(res.tables.values())[0]
corrected.head()